In [ ]:
import google.generativeai as genai
import pathlib
import httpx
import os

from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if API key is loaded
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please check your .env file.")

# Configure the API
genai.configure(api_key=api_key)

# Retrieve and encode the PDF byte
filepath = pathlib.Path('Input Data/Adbulla/PA.pdf')


In [9]:
from mistralai import Mistral

load_dotenv()
client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

uploaded = client.files.upload(
    file={"file_name": "referral_package.pdf", "content": open('Input Data/Adbulla/referral_package.pdf', "rb")},
    purpose="ocr"
)
signed = client.files.get_signed_url(file_id=uploaded.id)

ocr = client.ocr.process(
    model="mistral-ocr-latest",
    document={"type": "document_url", "document_url": signed.url},
    include_image_base64=False
)

# Save structured content in a variable
referral_package_text = ""
for pg in ocr.pages:
    referral_package_text += pg.markdown + "\n\n"

# Print structured content
for pg in ocr.pages:
    print(pg.markdown)


2023-05-30 11:01
EGAS Fax 2038183061 >> 16153783455
One Health Partners
P 1/10
Fax Referral To: (800) 223-4063
Email: intake@thealth.com
Date: 05/30/2023
Date of Birth: 02/17/1987
ICC-10 Code: 190
ICC-10 Code: 190
ICC-10 Code: 190
ICC-10 Code: 190
Date of Birth: 02/17/1987
Weight: 190 lbs OR kg
P 1/10
HEALTH PARTNERS
Therapy Status
New Start
☐ Continuing Therapy:
Last Dose:
PROVIDER INFORMATION
Ordering Provider: Timothy Adam, MD
Provider-Fax: 203-818-3061
Provider NPI: 1321124163
Provider Address: 2755 College Ave Ste. 100
Lessburg VA 20176
Provider Phone: 203-818-3060
MEDICATION ORDER
[x] Crohn's Disease Induction Phase:
Administer Skyrizi 600mg IV at week 0, week 4
and week 8 per protocol.
Refills x one year from
date of signature unless
indicated below.
[x] 1YR Refills
[x] Crohn's Disease Maintenance Phase:
Administer Skyrizi:
☐ 180mg SQ at week 12 and every 8 weeks
thereafter.
[x] 360mg SQ at week 12 and every 8 weeks
thereafter.
Please include the following lab results
required f

In [10]:
# Enhanced prompt for referral package analysis
referral_prompt = """You are analyzing a referral package (collection of scanned medical documents) to extract patient information that will be used to fill a Prior Authorization form.

CONTEXT: This referral package contains scanned documents like insurance cards, medical history notes, test results, and other supporting documentation. The extracted information will be mapped to specific fields in a PA form.

EXTRACTION FOCUS: Extract all available information that could be relevant for PA form completion, including:

PATIENT DEMOGRAPHICS:
- Full name, DOB, gender, address, phone numbers
- Patient ID numbers, MRN, account numbers
- Emergency contacts and relationships

INSURANCE INFORMATION:
- Insurance company name and plan details
- Member ID, group number, policy number
- Subscriber information and relationship to patient
- Coverage details and effective dates

CLINICAL INFORMATION:
- Primary and secondary diagnoses with ICD codes
- Current medications, dosages, and frequencies
- Allergies and adverse reactions
- Vital signs and lab results
- Previous treatments and outcomes
- Medical history and comorbidities

PROVIDER INFORMATION:
- Referring physician name, specialty, and contact info
- Practice name and address
- NPI numbers and license information
- Facility details where treatment will occur

TREATMENT DETAILS:
- Requested medication/procedure/device
- Dosage, frequency, duration
- Medical necessity justification
- Previous treatment failures
- Clinical criteria met for approval

SUPPORTING EVIDENCE:
- Lab results supporting diagnosis
- Imaging studies and results
- Functional assessments
- Specialist consultations
- Treatment response documentation

Return structured JSON with all extracted information, using null for missing data."""

# Extract structured data from referral package
model = genai.GenerativeModel('gemini-2.0-flash')
referral_response = model.generate_content([
    referral_prompt,
    f"REFERRAL PACKAGE OCR TEXT:\n{referral_package_text}"
])

referral_data = referral_response.text
print("Referral Package Data:")
print(referral_data)

Referral Package Data:
```json
{
  "PATIENT_DEMOGRAPHICS": {
    "full_name": "Akshay, chaudhari H",
    "dob": "02/17/1987",
    "gender": "Male",
    "age": "36",
    "address": "1460 El Camino Real, Arlington, VA-22407",
    "phone_numbers": {
      "home": "570-599-6973"
    },
    "patient_id": null,
    "mrn": null,
    "account_number": "D775152",
    "emergency_contacts": null,
    "relationship": null
  },
  "INSURANCE_INFORMATION": {
    "insurance_company_name": "Aetna Better Health of Virginia",
    "plan_details": null,
    "member_id": null,
    "group_number": null,
    "policy_number": "350410732018",
    "subscriber_information": "Akshay, Chaudhari H",
    "relationship_to_patient": "Self",
    "coverage_details": null,
    "effective_dates": null
  },
  "CLINICAL_INFORMATION": {
    "primary_diagnosis": "Crohn's disease of colon with rectal bleeding",
    "primary_diagnosis_icd_code": "K50.111",
    "secondary_diagnosis": "Anal disease consistent with Crohn's",
    "s

In [11]:
%pip install PyMuPDF
%pip install pdfplumber
%pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pdfplumber

# Load the PDF
with pdfplumber.open("Input Data/Adbulla/PA.pdf") as pdf:
    PA_cooridinates = ""
    for page_number, page in enumerate(pdf.pages, start=1):
        print(f"\n--- Page {page_number} ---")
        
        # Get all words with positional data
        words = page.extract_words()
        for word in words:
            text = word['text']
            x0 = word['x0']
            top = word['top']
            print(f"Text: {text}, X: {x0:.2f}, Y: {top:.2f}")
        # Extract text from the page
        PA_cooridinates += page.extract_text() + "\n"

        # Optionally, print the entire text on the page
        # print(page.extract_text())

    print (PA_cooridinates)
# Use the reportlab to fill in PA form with the extracted data from the referral package.



--- Page 1 ---
Text: MEDICARE, X: 171.12, Y: 18.50
Text: FORM, X: 260.94, Y: 18.50
Text: For, X: 445.38, Y: 18.16
Text: Medicare, X: 461.46, Y: 18.16
Text: Advantage, X: 500.63, Y: 18.16
Text: Part, X: 546.46, Y: 18.16
Text: B:, X: 565.30, Y: 18.16
Text: For, X: 445.38, Y: 27.94
Text: other, X: 461.47, Y: 27.94
Text: lines, X: 485.12, Y: 27.94
Text: of, X: 506.78, Y: 27.94
Text: business:, X: 517.17, Y: 27.94
Text: Riabni®, X: 171.12, Y: 34.60
Text: (rituximab-arrx),, X: 219.31, Y: 37.00
Text: Please, X: 445.38, Y: 37.29
Text: use, X: 473.76, Y: 37.29
Text: commercial, X: 489.83, Y: 37.29
Text: form., X: 535.67, Y: 37.29
Text: Rituxan®, X: 171.12, Y: 48.58
Text: (rituximab),, X: 227.23, Y: 50.98
Text: Ruxience, X: 301.58, Y: 50.98
Text: Note:, X: 445.38, Y: 51.56
Text: Riabni, X: 465.24, Y: 51.56
Text: and, X: 488.58, Y: 51.56
Text: Rituxan, X: 502.92, Y: 51.56
Text: are, X: 530.59, Y: 51.56
Text: non-, X: 543.01, Y: 51.56
Text: preferred., X: 445.38, Y: 59.66
Text: The, X: 479.99, Y:

In [13]:
import fitz  # PyMuPDF

doc = fitz.open("Input Data/Adbulla/PA.pdf")
pa_text_fitz = ""
for page in doc:
    text = page.get_text()
    pa_text_fitz += text + "\n\n"
# Print the extracted text
print(pa_text_fitz)

UMRMqFYNUWh
dvgB.IEOvRvOgSUSR.gSCBImCgSUSR.gSCBI
EbCBvDI#GuppGslyGsVy#u/kkzDIs##Lu
qd=DI
#GuuuGypsG>yssu
qCOI?v@SR.OvId@A.Bg.BvIE.OgICDI
y,HOzHqXzHqjH?kTOBHq[HD]HzIqGxBCq
!"#$%&%®'($%)*+"%&,-*./$&**0'
123%4*5%6+'7$242$5%8%4*5%6+'92:,2)5'
NOMT-h-’D-H-
F:,,qLkH,?zqC]zIq’HqTxC4,HIH?qO8?q,H1k’,HqLxBq4BHTHBIkLkTOIkx8qBHBkHCDhq
!"#$%#&'()'*$+#,!
!"#$%#!&'!#%($#)(*#+!!"#$%#!,$#(!
! -! ! ! ! ! -! ! !
!.&*#/*0$#/&*!&'!#1(%$234!5$#(!&'!6$7#!#%($#)(*#!
! -! !
! -! ! !
!-#*#-+'.'*$+'/(&0#12#%+#)&34,!! !
! 81&*(+!
!9$:+!
dD EdkEFmkIEmqGH?dkEGm
gbcpvu?Yf,#u
MYpvu?Yf,#u
.xU#u
-’’c,pp#u
(bv)#u
TvYv,#u
Rwq#u
/zf,uqjz;,#uu
<zc=uqjz;,#u
(,>>uqjz;,#uu
?fYb>#u
qYvb,;vu(@cc,;vu<,bAjv#u
>Bp uzc
=ApuuuqYvb,;vu/,bAjv#u
b;Cj,p zcuu
Cfpu ->>,cAb,p#
CD EmIJHdmKFIEmqGH?dkEGm
dvgB.I?vLMvOIENIO#u
POCQRIO#u
EBSQOv@#u
.z,puDYvb,;vujYE,uzvj,cuCzE,cYA,Fu
uG,pu
u?z
u
u
u
wHu),pIuDczEb’,uw.J#u
(Yccb,cu?Yf,#u
w;p@c,’#u
?v@SR.OvDu
uG,pu
u?zuuuwHu),pIuDczEb’,uw.uJ#uu
?v@SR.S@D
uG,p
u?z uwHu),pIuDczEb’,uw.uJ#uuu
KD EHFIKHECFH

In [14]:
prompt = """You are analyzing a Prior Authorization (PA) form PDF to understand its structure and extract form field information for automated filling.

CONTEXT: This is part of a healthcare automation pipeline where PA forms need to be filled using information from referral packages. PA forms are structured PDFs that may contain fillable form widgets (AcroForm fields) or be image-based forms.

TASK: Extract and analyze the PA form structure to identify:
1. All form fields that need to be filled
2. Field types (text, checkbox, dropdown, etc.)
3. Required vs optional fields
4. Conditional logic and field dependencies
5. Mutually exclusive options
6. Branching sections based on selections
7. Page numbers for accurate coordinate mapping

IMPORTANT CONSIDERATIONS:
- PA forms contain mutually exclusive options (e.g., "New Patient" vs "Existing Patient")
- Some sections are conditional and only relevant based on previous answers
- Not every field should be filled - only appropriate fields based on patient situation
- Focus on identifying fillable form widgets and their validation rules
- Include page numbers to ensure accurate coordinate mapping across multi-page forms

ANALYSIS STRUCTURE - Return as JSON:
{
    "form_metadata": {
        "form_type": "Prior Authorization",
        "drug_name": null,
        "insurance_company": null,
        "form_version": null,
        "total_pages": null,
        "has_fillable_widgets": null,
        "form_complexity": null
    },
    "form_fields": {
        "patient_information": {
            "required_fields": [],
            "optional_fields": [],
            "field_types": {},
            "validation_rules": {},
            "page_locations": {}
        },
        "provider_information": {
            "required_fields": [],
            "optional_fields": [],
            "field_types": {},
            "validation_rules": {},
            "page_locations": {}
        },
        "insurance_information": {
            "required_fields": [],
            "optional_fields": [],
            "field_types": {},
            "validation_rules": {},
            "page_locations": {}
        },
        "clinical_information": {
            "required_fields": [],
            "optional_fields": [],
            "field_types": {},
            "validation_rules": {},
            "page_locations": {}
        },
        "medication_details": {
            "required_fields": [],
            "optional_fields": [],
            "field_types": {},
            "validation_rules": {},
            "page_locations": {}
        },
        "medical_necessity": {
            "required_fields": [],
            "optional_fields": [],
            "field_types": {},
            "validation_rules": {},
            "page_locations": {}
        }
    },
    "form_logic": {
        "conditional_fields": {},
        "mutually_exclusive_groups": [],
        "branching_sections": {},
        "required_attachments": []
    },
    "fillable_widgets": {
        "text_fields": [],
        "checkboxes": [],
        "radio_buttons": [],
        "dropdowns": [],
        "signature_fields": [],
        "date_fields": []
    },
    "page_mapping": {
        "page_1": {
            "sections": [],
            "field_count": 0,
            "required_fields": []
        },
        "page_2": {
            "sections": [],
            "field_count": 0,
            "required_fields": []
        },
        "page_3": {
            "sections": [],
            "field_count": 0,
            "required_fields": []
        },
        "page_4": {
            "sections": [],
            "field_count": 0,
            "required_fields": []
        },
        "page_5": {
            "sections": [],
            "field_count": 0,
            "required_fields": []
        }
    },
    "completion_criteria": {
        "minimum_required_fields": [],
        "completion_percentage_estimate": null,
        "critical_missing_fields": []
    }
}

EXTRACTION GUIDELINES:
1. Focus on identifying actual fillable form fields, not just text content
2. Note field names, types, and any visible validation requirements
3. Identify sections that are mutually exclusive
4. Look for conditional logic indicators (e.g., "If yes, complete section B")
5. Distinguish between required fields (marked with *) and optional fields
6. Identify any pre-filled or locked fields
7. Note any signature requirements or date fields
8. Map each field to its corresponding page number for accurate coordinate placement
9. Identify which sections appear on which pages for proper form navigation

Return ONLY the JSON structure without markdown formatting."""

# Create the model
model = genai.GenerativeModel('gemini-2.0-flash')

# Generate content
response = model.generate_content([
    prompt,
    {
        "mime_type": "application/pdf",
        "data": filepath.read_bytes()
    }
])

# Store the response text
PA_form_analysis = response.text
# Print the response text
print("PA Form Analysis:")
print(response.text)

PA Form Analysis:
```json
{
    "form_metadata": {
        "form_type": "Prior Authorization",
        "drug_name": "Skyrizi (risankizumab-rzaa)",
        "insurance_company": "Aetna",
        "form_version": "2060 (9-23)",
        "total_pages": 2,
        "has_fillable_widgets": true,
        "form_complexity": "moderate"
    },
    "form_fields": {
        "patient_information": {
            "required_fields": [
                "First Name",
                "Last Name",
                "Address",
                "City",
                "State",
                "ZIP"
            ],
            "optional_fields": [
                "Home Phone",
                "Work Phone",
                "Cell Phone",
                "Patient Current Weight",
                "Patient Height",
                "Allergies",
                "DOB"
            ],
            "field_types": {
                "First Name": "text",
                "Last Name": "text",
                "Address": "text",
    

In [15]:
filling_prompt = """You are creating a form-filling strategy for a Prior Authorization form based on available patient data from a referral package.

TASK: Analyze the PA form structure and referral package data to create a mapping strategy that determines:
1. Which fields can be filled with available data
2. Which fields should remain blank (due to conditional logic or missing data)
3. How to handle mutually exclusive options
4. What information is missing and required

FORM FILLING RULES:
1. Only fill fields for which you have clear, accurate data
2. Respect mutually exclusive options (e.g., don't check both "New Patient" AND "Existing Patient")
3. Follow conditional logic (e.g., only fill dependent sections if trigger conditions are met)
4. Prioritize required fields over optional fields
5. Use exact text matching where possible
6. Format dates, phone numbers, and other data according to field requirements

MAPPING STRATEGY:
- Map referral package data to specific PA form fields
- Identify any data transformations needed
- Note confidence levels for each mapping
- Flag any ambiguous or unclear mappings

QUALITY ASSURANCE:
- Verify logical consistency of filled fields
- Ensure no conflicting information is entered
- Validate that conditional sections are appropriately completed
- Check that all required fields have been addressed

Return a detailed filling strategy as JSON:
{
    "filling_strategy": {
        "field_mappings": {},
        "conditional_logic_applied": {},
        "mutually_exclusive_selections": {},
        "data_transformations": {},
        "confidence_scores": {}
    },
    "completion_analysis": {
        "fillable_fields_count": null,
        "filled_fields_count": null,
        "completion_percentage": null,
        "critical_missing_fields": [],
        "optional_missing_fields": []
    },
    "form_values": {
        "patient_information": {},
        "provider_information": {},
        "insurance_information": {},
        "clinical_information": {},
        "medication_details": {},
        "medical_necessity": {}
    },
    "missing_information_report": {
        "required_but_missing": [],
        "recommended_but_missing": [],
        "could_not_determine": [],
        "data_quality_issues": []
    }
}"""

# Create form filling strategy
model = genai.GenerativeModel('gemini-2.0-flash')

strategy_response = model.generate_content([
    filling_prompt,
    f"PA FORM ANALYSIS:\n{PA_form_analysis}",
    f"REFERRAL PACKAGE DATA:\n{referral_package_text}"
])

filling_strategy = strategy_response.text
print("Form Filling Strategy:")
print(filling_strategy)

Form Filling Strategy:
```json
{
    "filling_strategy": {
        "field_mappings": {
            "patient_information": {
                "First Name": {
                    "source": "Progress Note P2/10: Akshay",
                    "confidence": 0.95,
                    "transformation": "extract first name"
                },
                "Last Name": {
                    "source": "Progress Note P2/10: chaudhari H",
                    "confidence": 0.95,
                    "transformation": "extract last name"
                },
                "Address": {
                    "source": "Progress Note P2/10: 1460 El Camino Real, Arlington, VA-22407",
                    "confidence": 0.95,
                    "transformation": null
                },
                "City": {
                    "source": "Address: Arlington",
                    "confidence": 0.95,
                    "transformation": "extract city"
                },
                "State": {
        

In [16]:
%pip install reportlab

Note: you may need to restart the kernel to use updated packages.


In [17]:
# Create form filling strategy

filling_prompt = """You are creating a precise coordinate-based form-filling strategy for a Prior Authorization form using extracted form field coordinates and available patient data.

TASK: Create an automated form filling system that maps referral package data to specific PDF coordinates to accurately populate form fields using ReportLab's canvas.drawString() method.

INPUT DATA:
1. PA_cooridinates: Contains extracted text with X,Y coordinates from the PDF form
2. Merge data: Contains previously analyzed referral package data and form structure

COORDINATE-BASED MAPPING REQUIREMENTS:
1. Use exact coordinate positions (X, Y) to place text in correct form fields
2. Respect field boundaries and text alignment requirements
3. Handle different field types (text boxes, checkboxes, radio buttons) with appropriate coordinate targeting
4. Ensure text placement doesn't overlap with form labels or borders

FORM FILLING STRATEGY:
- Map each data element to specific X,Y coordinates on the PDF
- Calculate optimal text placement within field boundaries
- Handle multi-line fields with proper line spacing
- Manage checkbox/radio button selection coordinates
- Apply proper text sizing and formatting for each field type

COORDINATE VALIDATION:
- Verify coordinates fall within expected form field areas
- Check for potential overlaps or misalignments
- Validate that text fits within field boundaries
- Ensure readability and professional appearance

RETURN FORMAT - JSON with coordinate-specific instructions compatible with ReportLab:
{
    "reportlab_instructions": [
        {
            "method": "drawString",
            "x": float,
            "y": float,
            "text": "value_to_place",
            "field_name": "descriptive_field_name",
            "font_size": int,
            "validation": "coordinate_check_results"
        },
        {
            "method": "rect",
            "x": float,
            "y": float,
            "width": float,
            "height": float,
            "stroke": 1,
            "fill": 1,
            "field_name": "checkbox_field_name"
        }
    ],
    "coordinate_mappings": {
        "field_name": {
            "coordinates": {"x": float, "y": float},
            "value": "text_to_place",
            "field_type": "text|checkbox|radio",
            "text_size": int,
            "alignment": "left|center|right",
            "validation": "coordinate_check_results"
        }
    },
    "placement_summary": {
        "total_fields_mapped": int,
        "text_fields_count": int,
        "checkbox_fields_count": int,
        "radio_button_fields_count": int
    },
    "quality_assurance": {
        "coordinate_conflicts": [],
        "boundary_violations": [],
        "alignment_issues": [],
        "missing_coordinates": []
    }
}

PRECISION REQUIREMENTS:
- Use coordinate data to ensure pixel-perfect placement
- Maintain a margin of error of no more than 1 pixel
- Ensure text is legible and fits within designated areas
- Generate ReportLab-compatible drawing commands
- Use available data from the referral package to fill the form fields accurately

REPORTLAB COMPATIBILITY:
- All coordinates should be compatible with canvas.drawString(x, y, text)
- For checkboxes, use canvas.rect(x, y, width, height, stroke=1, fill=1)
- Ensure proper font sizing for readability
- Account for PDF coordinate system (bottom-left origin)"""

# Use the existing PA.pdf file for form filling
import fitz
from reportlab.pdfgen import canvas
import json
import re

# Load the PA.pdf file for form filling
pdf_path = "Input Data/Adbulla/PA.pdf"

model = genai.GenerativeModel('gemini-2.0-flash')

strategy_response = model.generate_content([
    filling_prompt,
    f"PA FORM FIELD COORDINATES:\n{PA_cooridinates}",
    f"Merge data:\n{filling_strategy}"
])

print("Form Filling Strategy:")
print(strategy_response.text)

# Save the generated strategy to a file
with open("form_filling_strategy.json", "w") as f:
    f.write(strategy_response.text)

print("Form Filling Strategy saved to form_filling_strategy.json")

# Create a PDF file with ReportLab using the generated strategy
def create_filled_pdf(strategy_data, base_pdf_path, output_pdf):
    # Open the original PA.pdf as base
    base_doc = fitz.open(base_pdf_path)
    
    # Get page dimensions from the original PDF
    page_rect = base_doc[0].rect
    page_width = page_rect.width
    page_height = page_rect.height
    
    # Create a new PDF with ReportLab for overlaying text
    temp_overlay = "temp_overlay.pdf"
    c = canvas.Canvas(temp_overlay, pagesize=(page_width, page_height))
    
    # Iterate through the strategy instructions
    for instruction in strategy_data['reportlab_instructions']:
        if instruction['method'] == 'drawString':
            c.setFont("Helvetica", instruction['font_size'])
            c.drawString(instruction['x'], instruction['y'], instruction['text'])
        elif instruction['method'] == 'rect':
            c.rect(instruction['x'], instruction['y'], 
                  instruction.get('width', 6), instruction.get('height', 6), 
                  stroke=instruction['stroke'], fill=instruction['fill'])
    
    c.save()
    
    # Merge the overlay with the original PDF
    overlay_doc = fitz.open(temp_overlay)
    
    # For multi-page documents, overlay on the first page
    for page_num in range(len(base_doc)):
        base_page = base_doc[page_num]
        if page_num == 0:  # Apply overlay only to first page where coordinates are from
            overlay_page = overlay_doc[0]
            base_page.show_pdf_page(base_page.rect, overlay_doc, 0)
    
    # Save the result
    base_doc.save(output_pdf)
    
    # Clean up
    base_doc.close()
    overlay_doc.close()
    
    # Remove temp file
    import os
    if os.path.exists(temp_overlay):
        os.remove(temp_overlay)
    
    print(f"Filled PDF created: {output_pdf}")

# Load the strategy from the saved JSON file
with open("form_filling_strategy.json", "r") as f:
    content = f.read()

# Extract JSON from markdown code block
json_match = re.search(r'```json\n(.*?)\n```', content, re.DOTALL)
if json_match:
    json_content = json_match.group(1)
    strategy_data = json.loads(json_content)
else:
    # If no markdown blocks, try to parse as direct JSON
    try:
        strategy_data = json.loads(content)
    except json.JSONDecodeError:
        print("Error: Could not parse JSON from strategy response")
        strategy_data = None

# Create the filled PDF using the strategy and PA.pdf as base
if strategy_data:
    output_pdf = "filled_prior_authorization_form.pdf"
    create_filled_pdf(strategy_data, pdf_path, output_pdf)
    print(f"Filled PDF created at: {output_pdf}")
else:
    print("Could not create PDF due to JSON parsing error")

Form Filling Strategy:
```json
{
    "reportlab_instructions": [
        {
            "method": "drawString",
            "x": 78.0,
            "y": 583.0,
            "text": "Akshay",
            "field_name": "patient_first_name",
            "font_size": 8,
            "validation": "valid"
        },
        {
            "method": "drawString",
            "x": 195.0,
            "y": 583.0,
            "text": "Chaudhari",
            "field_name": "patient_last_name",
            "font_size": 8,
            "validation": "valid"
        },
        {
            "method": "drawString",
            "x": 297.0,
            "y": 583.0,
            "text": "02/17/1987",
            "field_name": "patient_dob",
            "font_size": 8,
            "validation": "valid"
        },
        {
            "method": "drawString",
            "x": 78.0,
            "y": 558.0,
            "text": "1460 El Camino Real",
            "field_name": "patient_address",
            "font_siz